# 07 - Performance Adjustment (Forecast Evaluation and Like-Game Review)

## Purpose
This notebook checks the demand and performance forecasts produced in notebook 06 against what actually happened. It:
* Splits forecasted games by release date into **train** (older 80%) and **test** (newest 20%).
* Lets you pick a test game and plot its forecast against actuals for cumulative sales, coin-in index and theo net win index.
* Plots the selected game against its **Eilers similar games** from notebook 05, so you can see whether the like games explain the forecast.
* Calculates error metrics (MAE, RMSE, bias, directional accuracy) for all test games and saves them to a table.

## Input / Output
| Variable | Table | Role |
| --- | --- | --- |
| `demand_path` | `development_042_silver_sandbox.demand_forecast.demand_forecast_results` | Demand (cumulative sales) forecasts from notebook 06 |
| `performance_path` | `development_042_silver_sandbox.demand_forecast.performance_forecast_results` | Performance index forecasts from notebook 06 |
| `ts_perf_path` | `development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series` | Actual Eilers monthly performance (class, supplier, indices, `no_of_slots`) |
| `like_games_path` | `development_042_silver_sandbox.demand_forecast.like_games_ai_attr` | Similar-game lists from notebook 05 |
| `sales_path` | `development_042_silver_sandbox.demand_forecast.sales_silver_comp` | Actual sales and lease records |
| `erro_table_path` | `development_042_silver_sandbox.demand_forecast.forecast_error_metrics` | **Output**: evaluation metrics (overwritten on every run) |

Widget: `test_game` - a dropdown of test games in the form `game name | target_id`, rebuilt each time the widget cell runs.

## Workflow by step (steps follow the code cells below, in order)
1. **Configuration** - define the table paths.
2. **Load data** - read all source tables (`abs_perf` and `ts_perf_df` both point to the Eilers time series) and display the forecasts and actuals.
3. **Preview like games** - display the similar-games table.
4. **Chronological train/test split**
   * Keep demand rows where `revenue_type` is sale/sales and normalise game names (trimmed, lower case, single spaces).
   * Keep sales targets with at least `MIN_MONTHS` (default 2) months of actual cumulative sales and a valid release date.
   * Keep only targets that also have a performance forecast with the same normalised name.
   * Rank by release date (newest first). The top 20% (rounded up) become **test** and the rest become **train**.
   * Produce `train_ids_df`, `test_ids_df`, `demand_train_df`, `demand_test_df`, `performance_train_df` and `performance_test_df`. Performance rows are given the matching sales `_target_id`.
   * Print how many games are in each split and how many were dropped (no performance match, no release date).
5. **Test-game widget** - rebuild the `test_game` dropdown from the current test set.
6. **Forecast vs. actual plot** - for the selected game, plot by month since release (1-12):
   * Predicted vs. actual cumulative sales, with the ±2 std band.
   * Predicted vs. actual coin-in index vs. house, with the ±2 std band.
   * Predicted vs. actual theo net win index vs. house, with the ±2 std band.
7. **Raw data check** - show Eilers performance, sales and like-game rows whose name contains the selected game name.
8. **Similar-game performance plot**
   * Look up the selected game's `eilers_similar_games` list.
   * For each similar game and for the selected game, calculate slot-weighted monthly averages of the two performance indices (rows with 1-5 slots only), aligned to each game's first Eilers month (months 1-12).
   * Plot the similar games (solid lines), the selected game's Eilers actuals (red dash-dot) and its forecast (black dashed).
9. **Ad-hoc lookup** - the same raw data check as step 7 for a hard-coded name (`cleopatra`).
10. **Error metrics (test games only)**
    * Demand: compare predicted and actual cumulative sales by month index.
    * Performance: rebuild actuals from the Eilers time series (slot-weighted by game and calendar month) and join them to forecasts on normalised name and `forecast_yearmonth`.
    * For each row, calculate the error, the absolute and squared error, and whether the predicted direction matches the actual direction from the previous month (consecutive months only).
    * Place each test game in a bracket (`top_20`, `middle_60`, `bottom_20`) by its actual value in its latest scored month, separately for each experiment and metric.
    * Calculate metrics at three levels: `per_game`, `overall` (`all_test_games`) and `bracket`. Overall and bracket metrics are pooled, so each game-month counts equally.
    * Overwrite `forecast_error_metrics` with the results.
11. Empty cell.

## Metric definitions
| Metric | Definition |
| --- | --- |
| `mae` | Mean absolute error, the average of abs(prediction - actual) |
| `rmse` | Square root of the mean squared error |
| `bias` | Average of (prediction - actual). Positive means the forecast is too high |
| `directional_accuracy` | Share of consecutive months where the predicted change has the same sign as the actual change |
| `valid_months` / `directional_comparisons` / `test_games` | Number of scored months, number of direction comparisons, number of distinct games |

## Output schema (`forecast_error_metrics`)
`result_level`, `forecast_type` (demand / performance), `metric_name`, `experiment_id`, `experiment_name`, `bracket`, `_target_id`, `target_game`, `test_games`, `valid_months`, `directional_comparisons`, `mae`, `rmse`, `bias`, `directional_accuracy`, `evaluation_split` (always `test`), `created_at`.

## Notes
* Run the cells in order. Step 6 needs the widget from step 5, and step 8 reuses `performance_game_pdf` from step 6 to draw the forecast line.
* Re-running step 4 can change the test set if the forecast tables have changed. If so, re-run step 5 to refresh the dropdown.
* The metrics table is completely overwritten on each run, so results from earlier runs are not kept unless they are still in the forecast tables.
* Despite the notebook's name, it evaluates and visualises the forecasts. It does not change them.
* Month alignment differs between charts. Step 6 uses the forecast `month_index` (months since release), while step 8 aligns games to their first available Eilers month.
* Steps 7 and 9 use substring matching (`contains`), so they may return other games whose names contain the search text.
* `ts_perf_path` is assigned twice in step 1, both times with the same value.
* In step 6, the `TEST_TARGET_ID is None` fallback never runs, because the ID always comes from the widget.

In [0]:
demand_path ="development_042_silver_sandbox.demand_forecast.demand_forecast_results"

performance_path ="development_042_silver_sandbox.demand_forecast.performance_forecast_results"
ts_perf_path='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'
erro_table_path = "development_042_silver_sandbox.demand_forecast.forecast_error_metrics"
like_games_path='development_042_silver_sandbox.demand_forecast.like_games_ai_attr'
ts_perf_path='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'# contains the class and, supplier, performance  the games in ts_perf are all found in game_attr_path
sales_path='development_042_silver_sandbox.demand_forecast.sales_silver_comp'# contains sales and lease of games
lease_path='development_042_silver_sandbox.demand_forecast.lease_units_bronz'# monthly rental observations
split_table_path = "development_042_silver_sandbox.demand_forecast.forecast_train_test_split"
SHOW_NEIGHBOR_CHARTS = False
DISTANCE_CUTOFF = 0.40  # Match notebook 06 for the optional neighbor review.
MIN_NEIGHBORS = 3


In [0]:
df_performance_forecast = spark.table(performance_path)
df_demand_forecast=spark.table(demand_path)
df_lease=spark.table(lease_path)
ts_perf_df = spark.table(ts_perf_path)
like_games_df=spark.table(like_games_path)
abs_sales=spark.table(sales_path)
abs_perf=spark.table(ts_perf_path)
display(df_demand_forecast)
display(df_performance_forecast)
display(ts_perf_df)
display(df_lease)

In [0]:

df_demand_forecast = spark.table(demand_path)
df_performance_forecast = spark.table(performance_path)

print(df_demand_forecast.columns)

In [0]:
display(like_games_df)

In [0]:
from collections import defaultdict
from datetime import date
import math
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType, LongType

MIN_MONTHS = 2
TEST_FRACTION = 0.20
SERIES_KEYS = ["experiment_id", "_target_id", "revenue_type"]

def normalized_game_name(column_name):
    return F.regexp_replace(F.lower(F.trim(F.col(column_name).cast("string"))), r"\s+", " ")

required_demand = {
    *SERIES_KEYS, "forecast_value", "actual_value", "actual_is_imputed",
    "metric_name", "curve_label", "own_status", "game_release_date",
}
missing = sorted(required_demand - set(df_demand_forecast.columns))
if missing:
    raise ValueError(f"Run the updated notebook 06 first. Missing demand columns: {missing}")

demand_all_df = (
    df_demand_forecast
    .where(F.col("revenue_type").isin("sales", "lease"))
    .withColumn("_game_key", normalized_game_name("target_game"))
    .withColumn("experiment_name", F.coalesce("experiment_name", "experiment_id"))
    .withColumn("_release_date", F.to_date("game_release_date"))
)
if demand_all_df.where(F.col("experiment_id").isNull()).limit(1).count():
    raise ValueError("Each forecast must have an experiment_id.")
if demand_all_df.groupBy(*SERIES_KEYS, "month_index").count().where(F.col("count") > 1).limit(1).count():
    raise ValueError("Duplicate demand rows per experiment/game/type/month. Rerun notebook 06.")

# Score only source-observed target months; filled values remain visible.
observed_pair = (
    F.col("forecast_value").isNotNull() & F.col("actual_value").isNotNull()
    & ~F.coalesce("actual_is_imputed", F.lit(False))
)
coverage_df = (
    demand_all_df.groupBy(*SERIES_KEYS)
    .agg(
        F.first("target_game").alias("target_game"),
        F.first("_game_key").alias("_game_key"),
        F.first("experiment_name").alias("experiment_name"),
        F.min("_release_date").alias("_sort_date"),
        F.countDistinct(F.when(observed_pair, F.col("month_index"))).alias("valid_months"),
        F.countDistinct(F.when(F.col("forecast_value").isNotNull(), F.col("month_index"))).alias("forecast_months"),
        F.countDistinct(F.when(F.col("actual_value").isNotNull(), F.col("month_index"))).alias("actual_months"),
    )
)
coverage_df = coverage_df.where((F.col("forecast_months") > 0) | (F.col("actual_months") > 0))
if coverage_df.limit(1).count() == 0:
    raise ValueError("No sales or lease forecast/actual curves are available. Run notebook 06 first.")
eligible_df = coverage_df.where(
    (F.col("valid_months") >= MIN_MONTHS) & F.col("_sort_date").isNotNull()
)

def assign_temporal_splits(records, test_fraction=0.20):
    """Split each experiment/type separately; release-date ties stay together."""
    if not 0 < test_fraction < 1:
        raise ValueError("test_fraction must be between zero and one.")
    cohorts = defaultdict(list)
    for item in records:
        row = dict(item)
        cohorts[(row["experiment_id"], row["revenue_type"])].append(row)
    output = []
    for cohort in cohorts.values():
        ordered = sorted(cohort, key=lambda r: (r["_sort_date"], r["_target_id"]), reverse=True)
        count = len(ordered)
        cutoff = ordered[max(1, math.ceil(count * test_fraction)) - 1]["_sort_date"]
        for row in ordered:
            output.append({
                **row,
                "split": "test" if row["_sort_date"] >= cutoff else "train",
                "split_cutoff_date": cutoff,
                "eligible_games_in_cohort": count,
            })
    return output

split_records = assign_temporal_splits([r.asDict() for r in eligible_df.collect()], TEST_FRACTION)
split_schema = StructType(eligible_df.schema.fields + [
    StructField("split", StringType(), False),
    StructField("split_cutoff_date", DateType(), False),
    StructField("eligible_games_in_cohort", LongType(), False),
])
eligible_split_df = spark.createDataFrame(split_records, split_schema)
# Curves with short histories remain selectable but are not evaluated.
game_split_df = (
    coverage_df
    .join(eligible_split_df.select(*SERIES_KEYS, "split", "split_cutoff_date", "eligible_games_in_cohort"), on=SERIES_KEYS, how="left")
    .withColumn("split", F.coalesce("split", F.lit("not_evaluated")))
    .withColumn("game_id", F.col("_target_id"))
)
train_ids_df = game_split_df.where(F.col("split") == "train")
test_ids_df = game_split_df.where(F.col("split") == "test")
demand_labeled_df = demand_all_df.join(game_split_df.select(*SERIES_KEYS, "split"), on=SERIES_KEYS, how="inner")
demand_train_df = demand_labeled_df.where(F.col("split") == "train")
demand_test_df = demand_labeled_df.where(F.col("split") == "test")
sales_train_df = demand_train_df.where(F.col("revenue_type") == "sales")
sales_test_df = demand_test_df.where(F.col("revenue_type") == "sales")
lease_train_df = demand_train_df.where(F.col("revenue_type") == "lease")
lease_test_df = demand_test_df.where(F.col("revenue_type") == "lease")

# Match by experiment, exact normalized game name, and revenue type.
# Use demand's ID in both outputs, even if input tables were regenerated.
performance_source_df = (
    df_performance_forecast
    .withColumn("own_status", normalized_game_name("own_status"))
    .where(F.col("own_status").isin("owned", "leased"))
    .withColumn("revenue_type", F.when(F.col("own_status") == "owned", "sales").otherwise("lease"))
    .withColumn("_game_key", normalized_game_name("target_game"))
    .withColumn("experiment_name", F.coalesce("experiment_name", "experiment_id"))
    .withColumnRenamed("_target_id", "_performance_target_id")
    .drop("game_id")
)
performance_labeled_df = (
    performance_source_df
    .join(
        game_split_df.select("experiment_id", "_game_key", "revenue_type", "_target_id", "split"),
        on=["experiment_id", "_game_key", "revenue_type"], how="inner",
    )
    .withColumn("game_id", F.col("_target_id"))
)
if performance_labeled_df.groupBy(*SERIES_KEYS, "month_index").count().where(F.col("count") > 1).limit(1).count():
    raise ValueError("Duplicate performance rows per experiment/game/type/month. Rerun notebook 06.")
performance_train_df = performance_labeled_df.where(F.col("split") == "train")
performance_test_df = performance_labeled_df.where(F.col("split") == "test")

game_split_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(split_table_path)
display(game_split_df.groupBy("experiment_id", "revenue_type", "split").count())
print("Splits are independent by type; a game's sales and rent can have different split labels.")
print("Release-date ties stay together; very small cohorts can have no training games.")


In [0]:
from pyspark.sql import functions as F

def build_game_options(records):
    options = {}
    labels = {"sales": "Sale", "lease": "Rent"}

    for record in records:
        row = dict(record)
        revenue_type = row["revenue_type"]

        if row["split"] != "test" or revenue_type not in labels:
            continue

        label = (
            f"{row['target_game']} | ID {row['_target_id']} | "
            f"{labels[revenue_type]} (test) | {row['experiment_id']}"
        )

        options[label] = {
            "_target_id": row["_target_id"],
            "experiment_id": row["experiment_id"],
            "target_game": row["target_game"],
            "revenue_types": [revenue_type],
            "splits": {revenue_type: "test"},
        }

    return dict(sorted(options.items()))


game_option_lookup = build_game_options([
    row.asDict()
    for row in game_split_df.where(
        (F.col("forecast_months") > 0)
        & (F.col("split") == "test")
        & F.col("revenue_type").isin("sales", "lease")
    ).collect()
])

game_options = list(game_option_lookup)

# Read the previous selection, then remove the old dropdown.
try:
    previous_choice = dbutils.widgets.get("test_game")
except Exception:
    previous_choice = None

try:
    dbutils.widgets.remove("test_game")
except Exception:
    pass

if not game_options:
    raise ValueError(
        "No eligible test sales or rent series has forecast_months > 0."
    )

default_choice = (
    previous_choice
    if previous_choice in game_option_lookup
    else game_options[0]
)

dbutils.widgets.dropdown(
    "test_game",
    default_choice,
    game_options,
    "Test game / Sale or Rent",
)


def selected_game_context():
    selection = dbutils.widgets.get("test_game")

    if selection not in game_option_lookup:
        raise ValueError("Rerun this cell to refresh the game choices.")

    return game_option_lookup[selection]

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_channel_review(demand_pdf, performance_pdf, context, revenue_type):
    """One portrait figure per commercial type, containing all three curves."""
    demand_pdf = demand_pdf.sort_values("month_index")
    performance_pdf = performance_pdf.sort_values("month_index") if not performance_pdf.empty else performance_pdf
    channel = "Sale" if revenue_type == "sales" else "Rent"
    ownership = "owned" if revenue_type == "sales" else "leased"
    fig, axes = plt.subplots(3, 1, figsize=(11, 13))
    color = "#1768AC" if revenue_type == "sales" else "#B45F06"

    def draw_panel(ax, frame, predicted, actual, lower, upper, title, ylabel, xlabel, imputed=None):
        ax.set_title(title, loc="left")
        ax.set_ylabel(ylabel)
        ax.set_xlabel(xlabel)
        ax.set_xticks(range(1, 13))
        ax.grid(True, alpha=0.2)
        if frame.empty:
            ax.text(0.5, 0.5, "No matching performance data", transform=ax.transAxes, ha="center")
            return
        x = pd.to_numeric(frame["month_index"], errors="coerce").to_numpy(dtype=float)
        def numbers(column):
            if column not in frame:
                return np.full(len(frame), np.nan)
            return pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)
        prediction, actual_values = numbers(predicted), numbers(actual)
        if np.isfinite(prediction).any():
            ax.plot(x, prediction, color=color, marker="o", label="Forecast")
            ax.fill_between(x, numbers(lower), numbers(upper), color=color, alpha=0.14, label="Neighbor variation (±2 SD)")
        else:
            ax.text(0.02, 0.95, "No usable neighbor forecast", transform=ax.transAxes, va="top")
        filled = frame[imputed].fillna(False).astype(bool).to_numpy() if imputed and imputed in frame else np.zeros(len(frame), dtype=bool)
        ax.plot(x, np.where(filled, np.nan, actual_values), color="#222222", marker="D", linestyle="--", label="Observed actual")
        if filled.any():
            ax.plot(x, actual_values, color="#666666", linestyle=":", alpha=0.7, label="Actual including internal fills")
            ax.scatter(x[filled], actual_values[filled], facecolors="none", edgecolors="#666666")
        ax.legend(fontsize=9)

    label = str(demand_pdf["curve_label"].iloc[0])
    draw_panel(
        axes[0], demand_pdf, "forecast_value", "actual_value", "low_band_2std", "high_band_2std",
        label, label, "Month since first sales observation" if revenue_type == "sales" else "Month since first rental observation",
        "actual_is_imputed",
    )
    for ax, metric, label in [
        (axes[1], "avg_coin_in_index_vs_house", "Coin-in index vs. house"),
        (axes[2], "avg_theo_net_win_index_vs_house", "Theo net win index vs. house"),
    ]:
        draw_panel(
            ax, performance_pdf, f"{metric}_mean", f"{metric}_actual", f"{metric}_lower_2std", f"{metric}_higher_2std",
            f"{label} — {ownership}", "Index", f"Month since first {ownership} Eilers observation",
        )
    fig.suptitle(
        f"{context['target_game']} | ID {context['_target_id']} | {channel} | {context['splits'][revenue_type].upper()}\n"
        f"Experiment: {context['experiment_id']}", fontsize=13,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    return fig

selected_context = selected_game_context()
TEST_TARGET_ID = selected_context["_target_id"]
selected_g = selected_context["target_game"]
selected_demand_by_type = {}
selected_performance_by_type = {}
for revenue_type in selected_context["revenue_types"]:
    condition = (
        (F.col("_target_id") == TEST_TARGET_ID)
        & (F.col("experiment_id") == selected_context["experiment_id"])
        & (F.col("revenue_type") == revenue_type)
    )
    demand_game_pdf = demand_labeled_df.where(condition).orderBy("month_index").toPandas()
    performance_game_pdf = performance_labeled_df.where(condition).orderBy("month_index").toPandas()
    selected_demand_by_type[revenue_type] = demand_game_pdf
    selected_performance_by_type[revenue_type] = performance_game_pdf
    if demand_game_pdf.empty:
        print(f"No {revenue_type} demand curve is available for {selected_g}.")
        continue
    plot_channel_review(demand_game_pdf, performance_game_pdf, selected_context, revenue_type)
    plt.show()


In [0]:
# Exact source-name checks for the selected experiment/game/type(s).
selected_context = selected_game_context()
selected_g = selected_context["target_game"]
selected_game_key = " ".join(selected_g.lower().split())
selected_like_df = like_games_df.where(
    (normalized_game_name("join_game_name") == selected_game_key)
    & (F.col("experiment_id") == selected_context["experiment_id"])
)
display(selected_like_df)
selected_metadata = demand_labeled_df.where(
    (F.col("_target_id") == selected_context["_target_id"])
    & (F.col("experiment_id") == selected_context["experiment_id"])
).first()
if selected_metadata is not None:
    for channel in selected_context["revenue_types"]:
        if channel == "sales":
            key = " ".join(str(selected_metadata["to_sales_name_match"]).lower().split())
            display(abs_sales.where(
                (normalized_game_name("normalized_game_name") == key)
                & normalized_game_name("revenue_type").isin("sale", "sales")
            ))
        else:
            key = " ".join(str(selected_metadata["to_rental_name_match"]).lower().split())
            display(df_lease.where(normalized_game_name("theme_name_eilers") == key))
        perf_name = selected_metadata["to_eilers_ts_name_match"] or selected_g
        status = "owned" if channel == "sales" else "leased"
        display(abs_perf.where(
            (normalized_game_name("game_name") == " ".join(perf_name.lower().split()))
            & (normalized_game_name("own_status") == status)
        ))


In [0]:
# Optional observed-neighbor review. Main sales/rent charts are in cell 7.
# Every neighbor review uses the selected experiment and ownership type.
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import json

if SHOW_NEIGHBOR_CHARTS:
    context = selected_game_context()
    matched = (
        like_games_df
        .where((normalized_game_name("join_game_name") == " ".join(context["target_game"].lower().split()))
               & (F.col("experiment_id") == context["experiment_id"]))
        .collect()
    )
    def parsed_neighbors(value):
        return json.loads(value) if isinstance(value, str) else (value or [])
    def neighbor_dict(item):
        return item.asDict() if hasattr(item, "asDict") else dict(item)
    def average_distance(row):
        values = [neighbor_dict(x).get("distance") for x in parsed_neighbors(row["eilers_similar_games"])]
        values = [x for x in values if x is not None]
        return sum(values) / len(values) if values else float("inf")
    like_row = min(matched, key=average_distance) if matched else None
    if like_row is None:
        print("No matching like-games row for this experiment.")
    else:
        candidates = [neighbor_dict(x) for x in parsed_neighbors(like_row["eilers_similar_games"])]
        candidates = sorted([x for x in candidates if x.get("distance") is not None and x.get("matched_source_name")], key=lambda x: (x["distance"], x["matched_source_name"]))
        kept = [x for x in candidates if x["distance"] <= DISTANCE_CUTOFF]
        kept += [x for x in candidates if x["distance"] > DISTANCE_CUTOFF][:max(0, MIN_NEIGHBORS - len(kept))]
        schema = StructType([
            StructField("_perf_key", StringType()), StructField("neighbor_game", StringType()),
            StructField("distance", DoubleType()),
        ])
        neighbor_rows = [(" ".join(x["matched_source_name"].lower().split()), x.get("join_game_name_key") or x["matched_source_name"], float(x["distance"])) for x in kept]
        neighbors_df = spark.createDataFrame(neighbor_rows, schema)
        for channel in context["revenue_types"]:
            status = "owned" if channel == "sales" else "leased"
            print(f"{channel}: commercial neighbors")
            commercial_column = "sales_similar_games" if channel == "sales" else "rental_similar_games"
            print(parsed_neighbors(like_row[commercial_column]))
            source = (
                abs_perf.where(normalized_game_name("own_status") == status)
                .withColumn("_perf_key", normalized_game_name("game_name"))
                .withColumn("_month", F.trunc(F.to_date("yearmonth"), "month"))
                .withColumn("_slots", F.col("no_of_slots").cast("double"))
                .where((F.col("_slots") > 0) & F.col("_month").isNotNull())
                .join(neighbors_df, on="_perf_key", how="inner")
                .withColumn("_start", F.min("_month").over(Window.partitionBy("_perf_key")))
                .withColumn("month_index", F.floor(F.months_between("_month", "_start")).cast("int") + 1)
                .where(F.col("month_index").between(1, 12))
            )
            metrics = ["avg_coin_in_index_vs_house", "avg_theo_net_win_index_vs_house"]
            by_slots = source.groupBy("neighbor_game", "month_index", "_slots").agg(*[F.avg(m).alias(m) for m in metrics])
            monthly = by_slots.groupBy("neighbor_game", "month_index").agg(*[
                (F.sum(F.col(m) * F.col("_slots")) / F.sum(F.when(F.col(m).isNotNull(), F.col("_slots")))).alias(m) for m in metrics
            ]).toPandas()
            if monthly.empty:
                print(f"No retained Eilers neighbors with {status} observations.")
                continue
            target = performance_labeled_df.where(
                (F.col("_target_id") == context["_target_id"])
                & (F.col("experiment_id") == context["experiment_id"])
                & (F.col("revenue_type") == channel)
            ).orderBy("month_index").toPandas()
            fig, axes = plt.subplots(2, 1, figsize=(11, 9))
            for ax, metric in zip(axes, metrics):
                for name, frame in monthly.groupby("neighbor_game"):
                    frame = frame.set_index("month_index").reindex(range(1, 13))
                    ax.plot(frame.index, frame[metric], alpha=0.6, label=name)
                if not target.empty:
                    ax.plot(target["month_index"], target[f"{metric}_mean"], color="black", linestyle="--", label="Target forecast")
                    ax.plot(target["month_index"], target[f"{metric}_actual"], color="red", marker="o", label="Target observed actual")
                ax.set_title(metric)
                ax.set_xlabel(f"Month since first {status} Eilers observation")
                ax.legend(fontsize=8)
                ax.grid(alpha=0.2)
            fig.suptitle(f"{context['target_game']} | ID {context['_target_id']} | {channel} | {status}")
            fig.tight_layout()
            plt.show()
else:
    print("Set SHOW_NEIGHBOR_CHARTS = True in cell 2 to add the neighbor review.")


In [0]:
# Selected game's independent train/test memberships.
context = selected_game_context()
display(game_split_df.where(
    (F.col("_target_id") == context["_target_id"])
    & (F.col("experiment_id") == context["experiment_id"])
).orderBy("revenue_type"))


In [0]:
# Evaluate sales demand, lease demand, and the corresponding performance
# separately. Only each type's test rows and observed actuals are scored.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

METRICS_TABLE = erro_table_path
identity_cols = ["forecast_type", "metric_name", "revenue_type", "own_status", "experiment_id", "experiment_name"]
game_metric_keys = identity_cols + ["_target_id"]

evaluation_rows_df = (
    demand_test_df
    .where(~F.coalesce("actual_is_imputed", F.lit(False)))
    .select(
        F.lit("demand").alias("forecast_type"), "metric_name", "revenue_type", "own_status",
        "_target_id", "target_game", "experiment_id", "experiment_name", "month_index",
        F.col("forecast_value").cast("double").alias("prediction"),
        F.col("actual_value").cast("double").alias("actual"),
    )
)
for metric in ["avg_coin_in_index_vs_house", "avg_theo_net_win_index_vs_house"]:
    frame = performance_test_df.select(
        F.lit("performance").alias("forecast_type"), F.lit(metric).alias("metric_name"),
        "revenue_type", "own_status", "_target_id", "target_game", "experiment_id", "experiment_name", "month_index",
        F.col(f"{metric}_mean").cast("double").alias("prediction"),
        # The unfilled actual column from notebook 06 is ownership-specific.
        F.col(f"{metric}_actual").cast("double").alias("actual"),
    )
    evaluation_rows_df = evaluation_rows_df.unionByName(frame)

evaluation_rows_df = evaluation_rows_df.where(
    F.col("prediction").isNotNull() & F.col("actual").isNotNull()
    & ~F.isnan("prediction") & ~F.isnan("actual")
    & (F.abs("prediction") != float("inf")) & (F.abs("actual") != float("inf"))
)
if evaluation_rows_df.limit(1).count() == 0:
    raise ValueError("No observed test forecast/actual pairs are available; the metrics table was not overwritten.")
series_window = Window.partitionBy(*game_metric_keys).orderBy("month_index")
evaluation_rows_df = (
    evaluation_rows_df
    .withColumn("error", F.col("prediction") - F.col("actual"))
    .withColumn("absolute_error", F.abs("error"))
    .withColumn("squared_error", F.pow("error", 2))
    .withColumn("_previous_month", F.lag("month_index").over(series_window))
    .withColumn("_previous_prediction", F.lag("prediction").over(series_window))
    .withColumn("_previous_actual", F.lag("actual").over(series_window))
)
predicted_change = F.col("prediction") - F.col("_previous_prediction")
actual_change = F.col("actual") - F.col("_previous_actual")
evaluation_rows_df = evaluation_rows_df.withColumn(
    "direction_correct",
    F.when(
        F.col("month_index") == F.col("_previous_month") + 1,
        (F.signum(predicted_change) == F.signum(actual_change)).cast("double"),
    ),
)

# Brackets use latest scored actuals independently per type/experiment/metric.
# This retains the original top-20 / middle-60 / bottom-20 convention.
latest_window = Window.partitionBy(*game_metric_keys).orderBy(F.col("month_index").desc())
game_actuals_df = (
    evaluation_rows_df.withColumn("_latest", F.row_number().over(latest_window))
    .where(F.col("_latest") == 1)
    .select(*game_metric_keys, "target_game", F.col("actual").alias("bracket_actual"))
)
bracket_window = Window.partitionBy(*identity_cols).orderBy(F.col("bracket_actual").desc(), "target_game", "_target_id")
bracket_count_window = Window.partitionBy(*identity_cols)
game_brackets_df = (
    game_actuals_df
    .withColumn("_rank", F.row_number().over(bracket_window))
    .withColumn("_games", F.count(F.lit(1)).over(bracket_count_window))
    .withColumn("_size", F.ceil(F.col("_games") * 0.20))
    .withColumn(
        "actual_bracket",
        F.when(F.col("_rank") <= F.col("_size"), "top_20")
        .when(F.col("_rank") > F.col("_games") - F.col("_size"), "bottom_20")
        .otherwise("middle_60"),
    )
    .select(*game_metric_keys, "actual_bracket")
)
evaluation_rows_df = evaluation_rows_df.join(game_brackets_df, on=game_metric_keys, how="left")

def metric_aggregations():
    return [
        F.count("actual").alias("valid_months"),
        F.count("direction_correct").alias("directional_comparisons"),
        F.countDistinct("_target_id").alias("test_games"),
        F.avg("absolute_error").alias("mae"),
        F.sqrt(F.avg("squared_error")).alias("rmse"),
        F.avg("error").alias("bias"),
        F.avg("direction_correct").alias("directional_accuracy"),
    ]

per_game_metrics_df = (
    evaluation_rows_df.groupBy(*game_metric_keys, "target_game", "actual_bracket")
    .agg(*metric_aggregations())
    .withColumn("result_level", F.lit("per_game"))
    .withColumnRenamed("actual_bracket", "bracket")
)

def aggregate_metrics(group_columns, result_level):
    return (
        evaluation_rows_df.groupBy(*group_columns).agg(*metric_aggregations())
        .withColumn("result_level", F.lit(result_level))
        .withColumn("_target_id", F.lit(None).cast("long"))
        .withColumn("target_game", F.lit(None).cast("string"))
    )

overall_metrics_df = aggregate_metrics(identity_cols, "overall").withColumn("bracket", F.lit("all_test_games"))
bracket_metrics_df = aggregate_metrics(identity_cols + ["actual_bracket"], "bracket").withColumnRenamed("actual_bracket", "bracket")
output_columns = [
    "result_level", *identity_cols, "bracket", "_target_id", "target_game", "test_games",
    "valid_months", "directional_comparisons", "mae", "rmse", "bias", "directional_accuracy",
]
forecast_error_metrics_df = (
    per_game_metrics_df.select(*output_columns)
    .unionByName(overall_metrics_df.select(*output_columns))
    .unionByName(bracket_metrics_df.select(*output_columns))
    .withColumn("game_id", F.col("_target_id"))
    .withColumn("evaluation_split", F.lit("test"))
    .withColumn("created_at", F.current_timestamp())
)
forecast_error_metrics_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(METRICS_TABLE)
display(forecast_error_metrics_df.orderBy("experiment_id", "revenue_type", "forecast_type", "metric_name", "result_level", "bracket", "target_game"))